# 01 — EDA: Histórico de Inventarios Herdez

Análisis exploratorio independiente para verificar los hallazgos documentados en `CLAUDE.md`.

**Objetivo:** Reproducir cada afirmación del CLAUDE.md con datos reales y reportar coincidencias/discrepancias.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Detectar raíz del proyecto (funciona desde notebooks/ y desde raíz)
PROJECT_ROOT = Path("..") if Path("../pyproject.toml").exists() else Path(".")
EXCEL_PATH = PROJECT_ROOT / "data" / "raw" / "herdez_inventario.xlsx.xlsx"

df = pd.read_excel(EXCEL_PATH, sheet_name="Historico_Inventarios")
df["Fecha"] = pd.to_datetime(df["Fecha"])

print(f"Shape: {df.shape}")
print(f"Rango fechas: {df['Fecha'].min().date()} → {df['Fecha'].max().date()}")
print(f"SKUs: {df['SKU_ID'].nunique()}, CEDIs: {df['CEDI'].nunique()}")
print(f"Nulls totales: {df.isnull().sum().sum()}")
df.head()

Shape: (1200, 11)
Rango fechas: 2024-03-01 → 2024-04-29
SKUs: 5, CEDIs: 4
Nulls totales: 0


,Fecha,SKU_ID,CEDI,Ventas_Unidades,Stock_Actual,Lead_Time_Dias,Promocion_Activa,Precio_Combustible_MXN,Clima,Costo_Quiebre_Stock_Diario,Costo_Transferencia_Unidad
0,2024-03-01,HZ-Salsa-Verde-200g,CEDI_Norte,195,1180,3,0,22.90,Despejado,15000,12.64
1,2024-03-01,HZ-Salsa-Verde-200g,CEDI_Sur,198,921,5,0,23.27,Despejado,15000,11.04
2,2024-03-01,HZ-Salsa-Verde-200g,CEDI_Occidente,185,71,5,0,23.51,Despejado,15000,10.07
3,2024-03-01,HZ-Salsa-Verde-200g,CEDI_Bajio,188,1007,5,0,24.78,Despejado,15000,10.89
4,2024-03-01,HZ-Salsa-Roja-200g,CEDI_Norte,204,216,3,0,23.55,Despejado,15000,11.16


## 1. Promocion_Activa: ¿tiene señal predictiva?

**CLAUDE.md dice:** ventas con promo (137.5) ≤ sin promo (140.5) → sin señal.

In [2]:
promo_stats = df.groupby("Promocion_Activa")["Ventas_Unidades"].agg(["mean", "median", "std", "count"])
print(promo_stats)
print(f"\nDiferencia de medias: {promo_stats.loc[1, 'mean'] - promo_stats.loc[0, 'mean']:.2f} unidades")
print(f"Conclusión: {'SIN señal (promo ≤ sin promo)' if promo_stats.loc[1, 'mean'] <= promo_stats.loc[0, 'mean'] else 'CON señal (promo > sin promo)'}")

                        mean  median        std  count
Promocion_Activa                                      
0                 140.477539   109.0  50.780492   1024
1                 137.465909   105.0  50.367665    176

Diferencia de medias: -3.01 unidades
Conclusión: SIN señal (promo ≤ sin promo)


## 2. Clima: ¿tiene señal predictiva?

**CLAUDE.md dice:** diferencias entre Despejado/Lluvia/Tormenta < 3%.

In [3]:
clima_stats = df.groupby("Clima")["Ventas_Unidades"].agg(["mean", "median", "count"])
print(clima_stats)

# Diferencia porcentual máxima entre categorías
max_mean = clima_stats["mean"].max()
min_mean = clima_stats["mean"].min()
pct_diff = (max_mean - min_mean) / min_mean * 100
print(f"\nDiferencia máxima entre categorías: {pct_diff:.2f}%")
print(f"Conclusión: {'SIN señal (< 3%)' if pct_diff < 3 else 'CON señal (≥ 3%)'}")

                 mean  median  count
Clima                               
Despejado  139.923345   108.0    861
Lluvia     139.008734   109.0    229
Tormenta   143.054545   115.0    110

Diferencia máxima entre categorías: 2.91%
Conclusión: SIN señal (< 3%)


## 3. Lead_Time_Dias: ¿constante por CEDI?

**CLAUDE.md dice:** Norte=3, resto=5.

In [4]:
lead_time_by_cedi = df.groupby("CEDI")["Lead_Time_Dias"].agg(["min", "max", "mean", "nunique"])
print(lead_time_by_cedi)
print(f"\nConclusión: {'Constante por CEDI' if (lead_time_by_cedi['nunique'] == 1).all() else 'VARÍA dentro de algún CEDI'}")

                min  max  mean  nunique
CEDI                                   
CEDI_Bajio        5    5   5.0        1
CEDI_Norte        3    3   3.0        1
CEDI_Occidente    5    5   5.0        1
CEDI_Sur          5    5   5.0        1

Conclusión: Constante por CEDI


## 4. Costo_Quiebre_Stock_Diario: ¿constante por SKU?

**CLAUDE.md dice:** $15k para salsas, $8k para el resto.

In [5]:
costo_by_sku = df.groupby("SKU_ID")["Costo_Quiebre_Stock_Diario"].agg(["min", "max", "nunique"])
print(costo_by_sku)
print(f"\nConclusión: {'Constante por SKU' if (costo_by_sku['nunique'] == 1).all() else 'VARÍA dentro de algún SKU'}")

                        min    max  nunique
SKU_ID                                     
HZ-Atun-Agua-140g      8000   8000        1
HZ-Champiñones-380g    8000   8000        1
HZ-Mole-Poblano-250g   8000   8000        1
HZ-Salsa-Roja-200g    15000  15000        1
HZ-Salsa-Verde-200g   15000  15000        1

Conclusión: Constante por SKU


## 5. Stock mínimo: ¿nunca llega a 0?

**CLAUDE.md dice:** Stock=0 nunca ocurre. Mínimo observado: 50 unidades.

In [6]:
stock_min = df["Stock_Actual"].min()
stock_zeros = (df["Stock_Actual"] == 0).sum()
print(f"Stock mínimo observado: {stock_min}")
print(f"Registros con stock=0: {stock_zeros}")
print(f"\nDistribución de Stock_Actual:")
print(df["Stock_Actual"].describe())

Stock mínimo observado: 50
Registros con stock=0: 0

Distribución de Stock_Actual:
count    1200.000000
mean     1044.578333
std       555.878399
min        50.000000
25%       567.000000
50%      1038.000000
75%      1518.000000
max      1999.000000
Name: Stock_Actual, dtype: float64


## 6. Distribución del target proyectado

El target NO está en los datos — es un proxy calculado:

```python
quiebre_proyectado = (stock_actual - ventas_promedio_7d * 5) < 0
```

In [7]:
# Calcular ventas_rolling_7d por grupo SKU-CEDI
df_sorted = df.sort_values(["SKU_ID", "CEDI", "Fecha"]).reset_index(drop=True)
df_sorted["ventas_rolling_7d"] = (
    df_sorted.groupby(["SKU_ID", "CEDI"])["Ventas_Unidades"]
    .transform(lambda x: x.rolling(7, min_periods=7).mean())
)

# Calcular target
df_sorted["quiebre_proyectado"] = (
    (df_sorted["Stock_Actual"] - df_sorted["ventas_rolling_7d"] * 5) < 0
).astype(int)

# Solo filas con rolling calculado
df_target = df_sorted.dropna(subset=["ventas_rolling_7d"])

print(f"Filas con target calculable: {len(df_target)}")
print(f"\nBalance de clases:")
print(df_target["quiebre_proyectado"].value_counts())
print(f"\n% quiebre: {df_target['quiebre_proyectado'].mean()*100:.1f}%")

Filas con target calculable: 1080

Balance de clases:
quiebre_proyectado
0    719
1    361
Name: count, dtype: int64

% quiebre: 33.4%


## 7. Correlaciones entre features numéricas

In [8]:
numeric_cols = ["Ventas_Unidades", "Stock_Actual", "Lead_Time_Dias",
               "Promocion_Activa", "Precio_Combustible_MXN",
               "Costo_Quiebre_Stock_Diario", "Costo_Transferencia_Unidad"]

corr = df[numeric_cols].corr()
print("Correlaciones con Ventas_Unidades:")
print(corr["Ventas_Unidades"].sort_values(ascending=False))
print("\nCorrelaciones con Stock_Actual:")
print(corr["Stock_Actual"].sort_values(ascending=False))

Correlaciones con Ventas_Unidades:
Ventas_Unidades               1.000000
Costo_Quiebre_Stock_Diario    0.971163
Costo_Transferencia_Unidad    0.028490
Lead_Time_Dias                0.004357
Promocion_Activa             -0.021019
Precio_Combustible_MXN       -0.028435
Stock_Actual                 -0.069056
Name: Ventas_Unidades, dtype: float64

Correlaciones con Stock_Actual:
Stock_Actual                  1.000000
Promocion_Activa              0.072565
Lead_Time_Dias                0.041581
Precio_Combustible_MXN       -0.029449
Costo_Transferencia_Unidad   -0.049831
Costo_Quiebre_Stock_Diario   -0.067465
Ventas_Unidades              -0.069056
Name: Stock_Actual, dtype: float64


## 8. Serie temporal: ventas y stock por SKU-CEDI

In [9]:
# Resumen por fecha (agregado)
daily_agg = df.groupby("Fecha").agg(
    ventas_total=("Ventas_Unidades", "sum"),
    stock_total=("Stock_Actual", "sum"),
    ventas_mean=("Ventas_Unidades", "mean"),
    stock_mean=("Stock_Actual", "mean"),
).reset_index()

print("Tendencia temporal (primeros y últimos 5 días):")
print(daily_agg[["Fecha", "ventas_mean", "stock_mean"]].head())
print("...")
print(daily_agg[["Fecha", "ventas_mean", "stock_mean"]].tail())

# ¿Hay tendencia clara?
ventas_trend = np.corrcoef(range(len(daily_agg)), daily_agg["ventas_mean"])[0, 1]
stock_trend = np.corrcoef(range(len(daily_agg)), daily_agg["stock_mean"])[0, 1]
print(f"\nCorrelación temporal (ventas vs día): {ventas_trend:.3f}")
print(f"Correlación temporal (stock vs día): {stock_trend:.3f}")

Tendencia temporal (primeros y últimos 5 días):
       Fecha  ventas_mean  stock_mean
0 2024-03-01       139.20      1195.1
1 2024-03-02       141.90      1072.5
2 2024-03-03       139.65      1165.4
3 2024-03-04       141.95       869.7
4 2024-03-05       140.00       923.2
...
        Fecha  ventas_mean  stock_mean
55 2024-04-25       139.95      958.20
56 2024-04-26       139.60     1041.45
57 2024-04-27       145.00      884.95
58 2024-04-28       142.10     1151.70
59 2024-04-29       140.50     1041.35

Correlación temporal (ventas vs día): 0.228
Correlación temporal (stock vs día): -0.018


## 9. Costo_Transferencia_Unidad: ¿la única variable de costo dinámica?

**CLAUDE.md dice:** es la única variable de costo que varía día a día.

In [10]:
# Variabilidad dentro de cada SKU-CEDI
costo_var = df.groupby(["SKU_ID", "CEDI"]).agg(
    costo_quiebre_nunique=("Costo_Quiebre_Stock_Diario", "nunique"),
    costo_transfer_nunique=("Costo_Transferencia_Unidad", "nunique"),
    costo_transfer_std=("Costo_Transferencia_Unidad", "std"),
)
print(costo_var)
print(f"\nCosto_Quiebre varía dentro de algún grupo: {(costo_var['costo_quiebre_nunique'] > 1).any()}")
print(f"Costo_Transferencia varía dentro de cada grupo: {(costo_var['costo_transfer_nunique'] > 1).all()}")

                                     costo_quiebre_nunique  \
SKU_ID               CEDI                                    
HZ-Atun-Agua-140g    CEDI_Bajio                          1   
                     CEDI_Norte                          1   
                     CEDI_Occidente                      1   
                     CEDI_Sur                            1   
HZ-Champiñones-380g  CEDI_Bajio                          1   
                     CEDI_Norte                          1   
                     CEDI_Occidente                      1   
                     CEDI_Sur                            1   
HZ-Mole-Poblano-250g CEDI_Bajio                          1   
                     CEDI_Norte                          1   
                     CEDI_Occidente                      1   
                     CEDI_Sur                            1   
HZ-Salsa-Roja-200g   CEDI_Bajio                          1   
                     CEDI_Norte                          1   
        

---

## Tabla comparativa: CLAUDE.md vs EDA

| Afirmación CLAUDE.md | Resultado EDA | Estado |
|---------------------|---------------|--------|
| Promocion_Activa sin señal (promo ≤ sin promo) | *ver sección 1* | |
| Clima sin señal (diferencia < 3%) | *ver sección 2* | |
| Lead_Time_Dias constante por CEDI (Norte=3, resto=5) | *ver sección 3* | |
| Costo_Quiebre constante por SKU ($15k salsas, $8k resto) | *ver sección 4* | |
| Stock=0 nunca ocurre (mínimo ~50) | *ver sección 5* | |
| Costo_Transferencia es la única variable de costo dinámica | *ver sección 9* | |

**Ejecutar este notebook para completar la tabla con los resultados reales.**